In [ ]:
import sys
import joblib
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.base import clone

PROJECT_ROOT = Path(r"D:\INSTANT\ML_SV_PROJ")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import (
    create_preprocessor,
    create_dense_preprocessor
)

from src.models import get_models

from src import evaluate_regression_model

split_data = joblib.load(
    ARTIFACTS_DIR / "train_test_split.joblib"
)

df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "vehicles_cleaned.csv"
)

X_train = split_data["X_train"]
X_test = split_data["X_test"]
y_train = split_data["y_train"]
y_test = split_data["y_test"]

preprocessor = joblib.load(
    ARTIFACTS_DIR / "preprocessor.joblib"
)

models = get_models()

print("Imports and artifacts loaded successfully.")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Available models:", len(models))

Imports and artifacts loaded successfully.
Training samples: 78165
Testing samples: 19542
Available models: 11


In [ ]:
numerical_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

model_results = {}

print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)
print("Model results dictionary initialized successfully.")

Numerical Features: ['year', 'mileage', 'tax', 'mpg', 'engineSize']
Categorical Features: ['model', 'transmission', 'fuelType', 'manufacturer']
Model results dictionary initialized successfully.


# 1 - Linear Regression

In [ ]:
linear_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", models["Linear Regression"])
])

print("Linear Regression pipeline created successfully.")

Linear Regression pipeline created successfully.


In [ ]:
linear_pipeline.fit(X_train, y_train)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


In [ ]:
linear_metrics, y_pred_linear = evaluate_regression_model(
    linear_pipeline,
    X_test,
    y_test
)

for metric, value in linear_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 2315.8865
RMSE: 3726.0594
R2: 0.8631
Median AE: 1565.3439
MAPE: 18.8248
Explained Variance: 0.8631


In [ ]:
model_results["Linear Regression"] = linear_metrics.copy()

# 2 - Ridge Regression

In [ ]:
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", models["Ridge"])
])

print("Ridge pipeline created successfully.")

Ridge pipeline created successfully.


In [ ]:
ridge_pipeline.fit(X_train, y_train)

print("Ridge Regression trained successfully.")

Ridge Regression trained successfully.


In [ ]:
ridge_metrics, y_pred_ridge = evaluate_regression_model(
    ridge_pipeline,
    X_test,
    y_test
)

for metric, value in ridge_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 2319.2634
RMSE: 3735.0320
R2: 0.8624
Median AE: 1568.5300
MAPE: 18.8419
Explained Variance: 0.8624


In [ ]:
model_results["Ridge"] = ridge_metrics.copy()

print("Ridge results saved successfully.")

Ridge results saved successfully.


In [ ]:
from sklearn.model_selection import GridSearchCV
ridge_param_grid = {
    "model__alpha": [0.01, 1, 100],
    "model__fit_intercept": [True, False],
    "model__solver": ["lsqr", "sag"],
}

ridge_grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

print("Ridge GridSearchCV created successfully.")

Ridge GridSearchCV created successfully.


In [ ]:
ridge_grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(ridge_grid_search.best_params_)

print("\nBest Cross-Validation R2:")
print(ridge_grid_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters:
{'model__alpha': 0.01, 'model__fit_intercept': True, 'model__solver': 'sag'}

Best Cross-Validation R2:
0.8622900801649103


In [ ]:
ridge_best_pipeline = ridge_grid_search.best_estimator_

ridge_tuned_metrics, y_pred_ridge_tuned = evaluate_regression_model(
    ridge_best_pipeline,
    X_test,
    y_test
)

for metric, value in ridge_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 2316.0646
RMSE: 3726.8154
R2: 0.8630
Median AE: 1565.6582
MAPE: 18.8208
Explained Variance: 0.8630


In [ ]:
model_results["Ridge (Default)"] = ridge_metrics.copy()
model_results["Ridge (Tuned)"] = ridge_tuned_metrics.copy()

print("Ridge results saved successfully.")

Ridge results saved successfully.


# 3 - Lasso Regression

In [ ]:
lasso_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", models["Lasso"])
])

print("Lasso Pipeline created successfully.")

Lasso Pipeline created successfully.


In [ ]:
lasso_pipeline.fit(X_train, y_train)

print("Lasso training completed successfully.")

Lasso training completed successfully.


d:\INSTANT\ML_SV_PROJ\.env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.967019e+10, tolerance: 7.534e+08
  model = cd_fast.sparse_enet_coordinate_descent(


In [ ]:
lasso_metrics, y_pred_lasso = evaluate_regression_model(
    lasso_pipeline,
    X_test,
    y_test
)

for metric, value in lasso_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 2345.0117
RMSE: 3766.4779
R2: 0.8601
Median AE: 1587.7255
MAPE: 19.0001
Explained Variance: 0.8601


In [ ]:
lasso_param_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__fit_intercept": [True, False]
}

In [ ]:
lasso_grid_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

lasso_grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(lasso_grid_search.best_params_)

print("\nBest CV R2:")
print(lasso_grid_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters:
{'model__alpha': 0.01, 'model__fit_intercept': True}

Best CV R2:
0.8622902886500707


d:\INSTANT\ML_SV_PROJ\.env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.585123e+11, tolerance: 7.534e+08
  model = cd_fast.sparse_enet_coordinate_descent(


In [ ]:
lasso_best_pipeline = lasso_grid_search.best_estimator_

lasso_tuned_metrics, y_pred_lasso_tuned = evaluate_regression_model(
    lasso_best_pipeline,
    X_test,
    y_test
)

for metric, value in lasso_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 2315.4227
RMSE: 3725.4774
R2: 0.8631
Median AE: 1566.5851
MAPE: 18.8158
Explained Variance: 0.8631


In [ ]:
lasso_coefficients = lasso_best_pipeline.named_steps["model"].coef_

total_features = len(lasso_coefficients)
zero_coefficients = np.sum(lasso_coefficients == 0)
selected_features = total_features - zero_coefficients

print("Total Features:", total_features)
print("Selected Features:", selected_features)
print("Zero Coefficients:", zero_coefficients)

Total Features: 216
Selected Features: 214
Zero Coefficients: 2


In [ ]:
model_results["Lasso (Default)"] = lasso_metrics.copy()
model_results["Lasso (Tuned)"] = lasso_tuned_metrics.copy()

print("Lasso results saved successfully.")

Lasso results saved successfully.


# 4 - Decision Tree

In [ ]:
tree_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", models["Decision Tree"])
])

print("Decision Tree Pipeline created successfully.")

Decision Tree Pipeline created successfully.


In [ ]:
tree_pipeline.fit(X_train, y_train)

print("Decision Tree training completed.")

Decision Tree training completed.


In [ ]:
tree_metrics, y_pred_tree = evaluate_regression_model(
    tree_pipeline,
    X_test,
    y_test
)

for metric, value in tree_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1456.0969
RMSE: 2534.6514
R2: 0.9366
Median AE: 955.0000
MAPE: 9.1034
Explained Variance: 0.9366


In [ ]:
tree_param_grid = {
    "model__max_depth": [10, 20, 30, None],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5]
}

In [ ]:
import os

os.makedirs(r"D:\joblib_temp", exist_ok=True)

os.environ["JOBLIB_TEMP_FOLDER"] = r"D:\joblib_temp"

In [ ]:
tree_grid_search = GridSearchCV(
    estimator=tree_pipeline,
    param_grid=tree_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

tree_grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(tree_grid_search.best_params_)

print("\nBest CV R2:")
print(tree_grid_search.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters:
{'model__max_depth': None, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}

Best CV R2:
0.9367053998478237


In [ ]:
tree_best_pipeline = tree_grid_search.best_estimator_

tree_tuned_metrics, y_pred_tree_tuned = evaluate_regression_model(
    tree_best_pipeline,
    X_test,
    y_test
)

for metric, value in tree_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1347.9889
RMSE: 2361.3883
R2: 0.9450
Median AE: 837.0000
MAPE: 8.4187
Explained Variance: 0.9450


In [ ]:
model_results["Decision Tree (Default)"] = tree_metrics.copy()
model_results["Decision Tree (Tuned)"] = tree_tuned_metrics.copy()

print("Decision Tree results saved successfully.")

Decision Tree results saved successfully.


# 5 - Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", models["Random Forest"])
])

print("Random Forest Pipeline created successfully.")

Random Forest Pipeline created successfully.


In [ ]:
rf_pipeline.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [ ]:
rf_metrics, y_pred_rf = evaluate_regression_model(
    rf_pipeline,
    X_test,
    y_test
)

for metric, value in rf_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1161.9989
RMSE: 1975.6072
R2: 0.9615
Median AE: 738.9150
MAPE: 7.3397
Explained Variance: 0.9615


In [ ]:
model_results["Random Forest (Default)"] = rf_metrics.copy()

# 6 - Extra Trees

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

extra_trees_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", ExtraTreesRegressor(
        random_state=42,
        n_jobs=2
    ))
])

In [ ]:
extra_trees_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['model','year','transmission',...,'mpg','engineSize','manufacturer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis c

In [ ]:
et_metrics, et_predictions = evaluate_regression_model(
    extra_trees_model,
    X_test,
    y_test
)

for metric, value in et_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1191.1743
RMSE: 1990.2467
R2: 0.9609
Median AE: 758.8950
MAPE: 7.5200
Explained Variance: 0.9609


In [ ]:
model_results["Extra Trees (Default)"] = et_metrics.copy()

# 7 - Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_boosting_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

In [ ]:
gradient_boosting_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['model','year','transmission',...,'mpg','engineSize','manufacturer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis c

In [ ]:
gb_metrics, gb_predictions = evaluate_regression_model(
    gradient_boosting_model,
    X_test,
    y_test
)

for metric, value in gb_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 2141.6430
RMSE: 3146.3051
R2: 0.9024
Median AE: 1542.0632
MAPE: 14.1907
Explained Variance: 0.9024


In [ ]:
gb_param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3],
    "model__min_samples_leaf": [1, 5]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

gb_grid_search = GridSearchCV(
    estimator=gradient_boosting_model,
    param_grid=gb_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=2,
    pre_dispatch=2,
    verbose=2
)

In [ ]:
from pathlib import Path
from joblib import parallel_config

TEMP_DIR = Path(r"D:\ML_TEMP")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from joblib import parallel_config
with parallel_config(
    backend="loky",
    n_jobs=-1,
    temp_folder=str(TEMP_DIR)
):
    gb_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [ ]:
print("Best Parameters:", gb_grid_search.best_params_)
print("Best CV R2:", gb_grid_search.best_score_)

Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Best CV R2: 0.9187981728344635


In [ ]:
best_gb_model = gb_grid_search.best_estimator_

In [ ]:
gb_tuned_metrics, gb_tuned_predictions = evaluate_regression_model(
    best_gb_model,
    X_test,
    y_test
)

for metric, value in gb_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1914.7886
RMSE: 2813.4584
R2: 0.9219
Median AE: 1370.6051
MAPE: 12.5017
Explained Variance: 0.9219


In [ ]:
model_results["Gradient Boosting (Default)"] = gb_metrics.copy()

model_results["Gradient Boosting (Tuned)"] = gb_tuned_metrics.copy()

# 8 - Hist Gradient Boosting

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from src.preprocessing import create_dense_preprocessor

dense_preprocessor = create_dense_preprocessor(
    numerical_features=numerical_features,
    categorical_features=categorical_features
)

In [ ]:
hist_gb_model = Pipeline([
    ("preprocessor", clone(dense_preprocessor)),
    ("model", HistGradientBoostingRegressor(
        random_state=42
    ))
])

In [ ]:
hist_gb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['model','year','transmission',...,'mpg','engineSize','manufacturer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0
,"remainder remainder: {'drop', 'passthrough'} or estimator, d

In [ ]:
hgb_metrics, hgb_predictions = evaluate_regression_model(
    hist_gb_model,
    X_test,
    y_test
)

for metric, value in hgb_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1452.0477
RMSE: 2298.0602
R2: 0.9479
Median AE: 1005.5853
MAPE: 9.3028
Explained Variance: 0.9479


In [ ]:
hgb_param_grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter": [100, 200],
    "model__max_leaf_nodes": [15, 31],
    "model__l2_regularization": [0, 1]
}

hgb_grid_search = GridSearchCV(
    estimator=hist_gb_model,
    param_grid=hgb_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=2,
    pre_dispatch=2,
    verbose=2
)

In [ ]:
with parallel_config(
    backend="loky",
    n_jobs=-1,
    temp_folder=str(TEMP_DIR)
):
    hgb_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [ ]:
print("Best Parameters:", hgb_grid_search.best_params_)
print("Best CV R2:", hgb_grid_search.best_score_)

Best Parameters: {'model__l2_regularization': 0, 'model__learning_rate': 0.1, 'model__max_iter': 200, 'model__max_leaf_nodes': 31}
Best CV R2: 0.9537611591259866


In [ ]:
best_hgb_model = hgb_grid_search.best_estimator_

In [ ]:
hgb_tuned_metrics, hgb_tuned_predictions = evaluate_regression_model(
    best_hgb_model,
    X_test,
    y_test
)

for metric, value in hgb_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1305.3658
RMSE: 2106.9159
R2: 0.9562
Median AE: 891.1478
MAPE: 8.2907
Explained Variance: 0.9562


In [ ]:
model_results["Hist Gradient Boosting (Default)"] = hgb_metrics.copy()

model_results["Hist Gradient Boosting (Tuned)"] = hgb_tuned_metrics.copy()

# 9 - XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    ))
])

In [ ]:
xgb_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['model','year','transmission',...,'mpg','engineSize','manufacturer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis c

In [ ]:
xgb_metrics, xgb_predictions = evaluate_regression_model(
    xgb_model,
    X_test,
    y_test
)

for metric, value in xgb_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1347.5488
RMSE: 2132.2700
R2: 0.9552
Median AE: 922.7954
MAPE: 8.6118
Explained Variance: 0.9552


In [ ]:
xgb_param_grid = {
    "model__n_estimators": [200, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [4, 6],
    "model__min_child_weight": [1, 5]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

xgb_grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    pre_dispatch=2,
    verbose=2
)

In [ ]:
with parallel_config(
    backend="loky",
    n_jobs=2,
    temp_folder=str(TEMP_DIR)
):
    xgb_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [ ]:
print("Best Parameters:", xgb_grid_search.best_params_)
print("Best CV R2:", xgb_grid_search.best_score_)

Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 6, 'model__min_child_weight': 1, 'model__n_estimators': 300}
Best CV R2: 0.9550327777862548


In [ ]:
best_xgb_model = xgb_grid_search.best_estimator_

xgb_tuned_metrics, xgb_tuned_predictions = evaluate_regression_model(
    best_xgb_model,
    X_test,
    y_test
)

for metric, value in xgb_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1341.8667
RMSE: 2132.0167
R2: 0.9552
Median AE: 915.2686
MAPE: 8.4903
Explained Variance: 0.9552


In [ ]:
model_results["XGBoost (Default)"] = xgb_metrics.copy()
model_results["XGBoost (Tuned)"] = xgb_tuned_metrics.copy()

# 10 - LightGBM

In [ ]:
from lightgbm import LGBMRegressor

lgbm_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LGBMRegressor(
        random_state=42,
        n_jobs=2,
        verbosity=-1
    ))
])

In [ ]:
lgbm_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['model','year','transmission',...,'mpg','engineSize','manufacturer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis c

In [ ]:
lgbm_metrics, lgbm_predictions = evaluate_regression_model(
    lgbm_model,
    X_test,
    y_test
)

for metric, value in lgbm_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1451.0911
RMSE: 2299.0563
R2: 0.9479
Median AE: 1006.3226
MAPE: 9.2924
Explained Variance: 0.9479


In [ ]:
lgbm_param_grid = {
    "model__n_estimators": [200, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__num_leaves": [31, 50],
    "model__max_depth": [-1, 10]
}

In [ ]:
from sklearn.model_selection import GridSearchCV

lgbm_grid_search = GridSearchCV(
    estimator=lgbm_model,
    param_grid=lgbm_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=2,
    pre_dispatch=2,
    verbose=2
)

In [ ]:
with parallel_config(
    backend="loky",
    n_jobs=2,
    temp_folder=str(TEMP_DIR)
):
    lgbm_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [ ]:
print("Best Parameters:", lgbm_grid_search.best_params_)
print("Best CV R2:", lgbm_grid_search.best_score_)

Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': -1, 'model__n_estimators': 300, 'model__num_leaves': 50}
Best CV R2: 0.9617449330568464


In [ ]:
best_lgbm_model = lgbm_grid_search.best_estimator_

lgbm_tuned_metrics, lgbm_tuned_predictions = evaluate_regression_model(
    best_lgbm_model,
    X_test,
    y_test
)

for metric, value in lgbm_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1176.8073
RMSE: 1938.8164
R2: 0.9629
Median AE: 786.3559
MAPE: 7.4203
Explained Variance: 0.9629


In [ ]:
model_results["LightGBM (Default)"] = lgbm_metrics.copy()
model_results["LightGBM (Tuned)"] = lgbm_tuned_metrics.copy()

# 11 - CatBoost

In [ ]:
from catboost import CatBoostRegressor

catboost_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", CatBoostRegressor(
        random_seed=42,
        verbose=0,
        thread_count=2
    ))
])

In [ ]:
catboost_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](0,)",[]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['model','year','transmission',...,'mpg','engineSize','manufacturer']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remainin

In [ ]:
catboost_metrics, catboost_predictions = evaluate_regression_model(
    catboost_model,
    X_test,
    y_test
)

for metric, value in catboost_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1257.9270
RMSE: 2008.0248
R2: 0.9602
Median AE: 860.6192
MAPE: 8.0885
Explained Variance: 0.9602


In [ ]:
catboost_param_grid = {
    "model__iterations": [300, 500],
    "model__learning_rate": [0.05, 0.1],
    "model__depth": [6, 8],
    "model__l2_leaf_reg": [3, 5]
}

In [ ]:
catboost_grid_search = GridSearchCV(
    estimator=catboost_model,
    param_grid=catboost_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=2,
    pre_dispatch=2,
    verbose=2
)

In [ ]:
with parallel_config(
    backend="loky",
    n_jobs=2,
    temp_folder=str(TEMP_DIR)
):
    catboost_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [ ]:
print("Best Parameters:", catboost_grid_search.best_params_)
print("Best CV R2:", catboost_grid_search.best_score_)

Best Parameters: {'model__depth': 8, 'model__iterations': 500, 'model__l2_leaf_reg': 3, 'model__learning_rate': 0.1}
Best CV R2: 0.9589269124902428


In [ ]:
best_catboost_model = catboost_grid_search.best_estimator_

catboost_tuned_metrics, catboost_tuned_predictions = evaluate_regression_model(
    best_catboost_model,
    X_test,
    y_test
)

for metric, value in catboost_tuned_metrics.items():
    print(f"{metric}: {value:.4f}")

MAE: 1258.6575
RMSE: 2010.7300
R2: 0.9601
Median AE: 869.2228
MAPE: 8.1170
Explained Variance: 0.9601


In [ ]:
model_results["CatBoost (Default)"] = catboost_metrics.copy()
model_results["CatBoost (Tuned)"] = catboost_tuned_metrics.copy()

# Model Comparsion

In [ ]:
results_df = pd.DataFrame(model_results).T

results_df = results_df.sort_values(
    by="R2",
    ascending=False
)

results_df

,MAE,RMSE,R2,Median AE,MAPE,Explained Variance
LightGBM (Tuned),1176.807312,1938.816421,0.962928,786.355926,7.420314,0.962929
Random Forest (Default),1161.998864,1975.607200,0.961508,738.915000,7.339663,0.961509
Extra Trees (Default),1191.174276,1990.246717,0.960935,758.895000,7.519957,0.960935
CatBoost (Default),1257.926953,2008.024765,0.960234,860.619166,8.088457,0.960234
CatBoost (Tuned),1258.657524,2010.729974,0.960127,869.222810,8.116965,0.960127
Hist Gradient Boosting (Tuned),1305.365822,2106.915889,0.956221,891.147809,8.290741,0.956222
XGBoost (Tuned),1341.866699,2132.016651,0.955171,915.268555,8.490258,0.955172
XGBoost (Default),1347.548828,2132.270034,0.955161,922.795410,8.611816,0.955161
Hist Gradient Boosting (Default),1452.047704,2298.060154,0.947917,1005.585317,9.302758,0.947918
LightGBM (Default),1451.091069,2299.056269,0.947872,1006.322593,9.292400,0.947873


In [ ]:
results_df.to_csv(
    ARTIFACTS_DIR / "model_comparison.csv",
    index=True
)

In [ ]:
best_model_name = results_df["R2"].idxmax()

print("Best Model:", best_model_name)
print(results_df.loc[best_model_name])

Best Model: LightGBM (Tuned)
MAE                   1176.807312
RMSE                  1938.816421
R2                       0.962928
Median AE              786.355926
MAPE                     7.420314
Explained Variance       0.962929
Name: LightGBM (Tuned), dtype: float64


In [ ]:
import joblib

final_model = lgbm_grid_search.best_estimator_

joblib.dump(
    final_model,
    ARTIFACTS_DIR / "final_model.joblib"
)

['D:\\INSTANT\\ML_SV_PROJ\\artifacts\\final_model.joblib']

In [ ]:
import pandas as pd

feature_importance = pd.Series(
    final_model.named_steps["model"].feature_importances_,
    index=final_model.named_steps["preprocessor"].get_feature_names_out()
).sort_values(ascending=False)

feature_importance.head(15)

num__mpg                       3353
num__mileage                   2197
num__engineSize                1332
num__year                      1321
num__tax                        539
cat__transmission_Manual        199
cat__manufacturer_mercedes      195
cat__transmission_Automatic     187
cat__manufacturer_bmw           153
cat__manufacturer_audi          152
cat__transmission_Semi-Auto     127
cat__fuelType_Diesel            120
cat__fuelType_Petrol            118
cat__model_C Class              107
cat__model_2 Series              96
dtype: int32